# LinguoMT — African-Celtic × Whisper+NLLB

| | |
|---|---|
| **Model** | Whisper-large-v3 + NLLB-200-distilled-600M (cascade ASR+MT) |
| **Dataset** | McGill-NLP/african_celtic_dataset (IWSLT 2026) |
| **Languages** | Yoruba · Hausa  *(Igbo excluded: no Whisper language token)* |
| **Papers** | Paper 1 (benchmark), Paper 4 (cascade) |
| **DEBUG runtime** | ~10 min |
| **FULL runtime** | ~30–60 min |

Runs this single experiment, consolidates its outputs, then generates a Markdown
results report with tables, cross-model comparisons, SOTA analysis, and discussion.

> **Before running:** Runtime → Change runtime type → **T4 GPU** (or A100).
> **Quick start:** Configure Step 4, then **Runtime → Run all**.

## Step 1 — Mount Google Drive

Outputs and the results report are backed up to `MyDrive/LinguoMT-AfricaS2T/` at the end of the run.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted.")
except ImportError:
    print("Not in Colab — Drive mount skipped.")

## Step 2 — Clone / Update Repository

In [ ]:
import os, subprocess

REPO_DIR = "/content/LinguoMT-AfricaS2T"
REPO_URL = "https://github.com/prsisda/LinguoMT-AfricaS2T.git"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
    print("Repo updated to origin/main")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Step 3 — Install Dependencies

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U",
    "transformers>=4.40", "datasets", "sacrebleu", "librosa", "soundfile",
    "sentencepiece", "accelerate", "jiwer", "pandas==2.2.2",
    "pyarrow>=15.0.0", "protobuf", "tabulate",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchcodec",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121"], check=False)
print("\nDependencies ready.")

## Step 4 — Configure Paper Mode & Settings

| Variable | Values | Notes |
|---|---|---|
| `PAPER_MODE` | `benchmark` · `adaptation` · `audio` · `cascade` · `transfer` | Which paper to produce |
| `DEBUG_MODE` | `True` / `False` | `True` ≈ ~10 min · `False` ≈ ~30–60 min |
| `ENABLE_FINETUNING` | `True` / `False` | Papers 2 & 3 only |
| `SCALING_BUDGETS` | `[100,500,1000,0]` or `[]` | Paper 2 only |
| `SOTA_FILE` | path or `""` | CSV with system/language/BLEU/venue/year columns |

`EXPERIMENT` is fixed to `AfricanCeltic__WhisperNLLB` for this notebook.

In [ ]:
import re, pathlib

# ── SELECT PAPER (uncomment exactly one) ──────────────────────────────
PAPER_MODE = "benchmark"    # Paper 1 — zero-shot baselines           ← ACTIVE
# PAPER_MODE = "adaptation" # Paper 2 — fine-tuning comparison
# PAPER_MODE = "audio"      # Paper 3 — audio strategy analysis
# PAPER_MODE = "cascade"    # Paper 4 — cascade vs end-to-end
# PAPER_MODE = "transfer"   # Paper 5 — cross-lingual transfer
# ─────────────────────────────────────────────────────────────────────

# ── RUN MODE ──────────────────────────────────────────────────────────
DEBUG_MODE = True   # True ≈ ~10 min | False ≈ ~30–60 min
# ─────────────────────────────────────────────────────────────────────

# ── FINE-TUNING (Papers 2 & 3 only) ──────────────────────────────────
ENABLE_FINETUNING = False
FINETUNING_METHOD = "lora"   # lora | adapter | full
# ─────────────────────────────────────────────────────────────────────

# ── DATA SCALING BUDGETS (Paper 2 only) ──────────────────────────────
# e.g. SCALING_BUDGETS = [100, 500, 1000, 0]   # 0 = full train set
SCALING_BUDGETS = []
# ─────────────────────────────────────────────────────────────────────

# ── SOTA FILE (optional) ──────────────────────────────────────────────
# e.g. "sota/paper1_benchmark/sota_results.csv"  — leave "" to skip
SOTA_FILE = ""
# ─────────────────────────────────────────────────────────────────────

# Experiment is fixed for this notebook
EXPERIMENT = "AfricanCeltic__WhisperNLLB"
EXPERIMENT_FAMILY = "AfricanCeltic__WhisperNLLB"
SCRIPT  = pathlib.Path(f"{EXPERIMENT}/notebooks/run_experiment.py")
SCRIPTS = {EXPERIMENT: SCRIPT}

print(f"Experiment : {EXPERIMENT}")
print(f"Paper      : {PAPER_MODE}")
print(f"Mode       : {'DEBUG  (fast test)' if DEBUG_MODE else 'FULL   (paper run)'}")
print(f"Fine-tune  : {ENABLE_FINETUNING}  (method: {FINETUNING_METHOD})")
print(f"Scaling    : {SCALING_BUDGETS if SCALING_BUDGETS else 'disabled'}")
print(f"SOTA file  : {SOTA_FILE or 'disabled'}")

## Step 5 — Patch Script & Run Experiment

Pulls the latest framework code, applies your configuration, and runs the `AfricanCeltic__WhisperNLLB` pipeline.

Output is written to `/content/outputs/<timestamp_slug>/`.

In [ ]:
import subprocess, re

subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)
print("Repository updated.")

mode_str    = "True" if DEBUG_MODE else "False"
ft_str      = "True" if ENABLE_FINETUNING else "False"
budgets_str = repr(SCALING_BUDGETS)

for name, sp in SCRIPTS.items():
    src = sp.read_text()
    src = re.sub(r"(?m)^(DEBUG_MODE\s*=\s*)(True|False)",             rf"\g<1>{mode_str}",           src)
    src = re.sub(r'(?m)^(PAPER_MODE\s*=\s*)["\'][^"\']+["\']',        rf'\g<1>"{PAPER_MODE}"',        src)
    src = re.sub(r"(?m)^(ENABLE_FINETUNING\s*=\s*)(True|False)",      rf"\g<1>{ft_str}",              src)
    src = re.sub(r"(?m)^(FINETUNING_METHOD\s*=\s*)['\"][^'\"]+['\"]", rf"\g<1>'{FINETUNING_METHOD}'", src)
    src = re.sub(r"(?m)^(SCALING_BUDGETS\s*=\s*)\[[^\]]*\]",          rf"\g<1>{budgets_str}",         src)
    src = re.sub(r'(?m)^(SOTA_FILE\s*=\s*)["\'][^"\']*["\']',         rf'\g<1>"{SOTA_FILE}"',         src)
    sp.write_text(src)
    print(f"  Patched: {name}")

print(f"\nScripts configured — {PAPER_MODE} | {'DEBUG' if DEBUG_MODE else 'FULL'}")

In [ ]:
import sys, subprocess as _sp

for name, sp in SCRIPTS.items():
    print(f"\n{'='*64}\n  Running : {name}\n  Paper   : {PAPER_MODE}  |  Mode : {'DEBUG' if DEBUG_MODE else 'FULL'}\n{'='*64}\n")
    result = _sp.run([sys.executable, str(sp)])
    if result.returncode != 0:
        raise RuntimeError(f"{name} failed (exit code {result.returncode})")

print("\nAll experiments finished.")

## Step 6 — Consolidate Metrics

Merges metric CSVs from this experiment's output into a consolidated directory.

In [ ]:
import json as _json
import pandas as pd
from pathlib import Path
from datetime import datetime

out_root         = Path("/content/outputs")
consolidated_dir = out_root / f"consolidated_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
consolidated_dir.mkdir(parents=True, exist_ok=True)

all_text, all_asr, all_audio = [], [], []

for run_dir in sorted(out_root.glob("*/")):
    if "consolidated" in run_dir.name:
        continue
    exp_label = run_dir.name
    cfg_path  = run_dir / "config.json"
    if cfg_path.exists():
        exp_label = _json.loads(cfg_path.read_text()).get("experiment_family", run_dir.name)
    for fname, store in [
        ("text_metrics.csv",  all_text),
        ("asr_metrics.csv",   all_asr),
        ("audio_metrics.csv", all_audio),
    ]:
        fpath = run_dir / "metrics" / fname
        if fpath.exists():
            df = pd.read_csv(fpath)
            if "experiment" not in df.columns:
                df.insert(0, "experiment", exp_label)
            store.append(df)

summary_parts = [f"# LinguoMT Consolidated Metrics — {PAPER_MODE}\n\n"]
for label, store, out_name in [
    ("Text Translation", all_text,  "text_metrics_all.csv"),
    ("ASR",              all_asr,   "asr_metrics_all.csv"),
    ("Audio",            all_audio, "audio_metrics_all.csv"),
]:
    if store:
        merged = pd.concat(store, ignore_index=True)
        merged.to_csv(consolidated_dir / out_name, index=False)
        summary_parts += [f"## {label}\n\n", merged.to_markdown(index=False), "\n\n"]
        print(f"=== {label} ===\n{merged.to_string(index=False)}\n")

(consolidated_dir / "metrics_summary.md").write_text("".join(summary_parts))
print(f"Consolidated → {consolidated_dir}")

## Step 7 — Generate Results Report

Reads the consolidated metrics and produces `papers/<paper_id>/results_report.md` — a structured Markdown document with results tables, SOTA comparison, and discussion.

> Results for other experiments in this paper will be absent until you run their notebooks.
> The report shows what is available and marks missing data clearly.

In [ ]:
import subprocess, sys
from pathlib import Path
from pathlib import Path

out_root = Path("/content/outputs")
cons = sorted(out_root.glob("consolidated_*/"), reverse=True)
if not cons:
    print("ERROR: No consolidated directory found — run Step 6 first.")
else:
    consolidated_dir = cons[0]
    cmd = [
        sys.executable, "papers/generate_report.py", PAPER_MODE,
        "--consolidated-dir", str(consolidated_dir),
    ]
    if SOTA_FILE:
        cmd += ["--sota-file", SOTA_FILE]

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    else:
        paper_id_map = {
            "benchmark":  "paper1_benchmark",
            "adaptation": "paper2_adaptation",
            "audio":      "paper3_audio",
            "cascade":    "paper4_cascade",
            "transfer":   "paper5_transfer",
        }
        report_path = Path(f"papers/{paper_id_map[PAPER_MODE]}/results_report.md")
        if report_path.exists():
            text    = report_path.read_text()
            preview = text[:5000]
            print("\n" + "="*64)
            print(f"REPORT PREVIEW  —  {report_path}")
            print("="*64)
            print(preview)
            if len(text) > 5000:
                print(f"\n... ({len(text) - 5000} more chars — open the file for the complete report)")

## Step 8 — Package & Download

Zips the report, consolidated metrics, tables, plots, and interpretations. Saves to Drive and triggers download.

In [ ]:
import shutil, json as _json
from pathlib import Path
from datetime import datetime

try:
    from google.colab import files as _colab_files
    _in_colab = True
except ImportError:
    _in_colab = False

ts        = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
repo_root = Path("/content/LinguoMT-AfricaS2T")
mode_tag  = "debug" if DEBUG_MODE else "full"
paper_id_map = {
    "benchmark":  "paper1_benchmark",
    "adaptation": "paper2_adaptation",
    "audio":      "paper3_audio",
    "cascade":    "paper4_cascade",
    "transfer":   "paper5_transfer",
}
PAPER_ID = paper_id_map[PAPER_MODE]
pkg_name  = f"linguomt_{PAPER_ID}_{mode_tag}_{ts}"
pkg_dir   = Path("/content") / pkg_name
pkg_dir.mkdir(parents=True, exist_ok=True)

# 1. Results report (Markdown)
report_md = repo_root / "papers" / PAPER_ID / "results_report.md"
if report_md.exists():
    shutil.copy2(str(report_md), str(pkg_dir / "results_report.md"))

# 2. Consolidated metrics
out_root = Path("/content/outputs")
cons = sorted(out_root.glob("consolidated_*/"), reverse=True)
if cons:
    shutil.copytree(str(cons[0]), str(pkg_dir / "consolidated_metrics"))

# 3. Per-experiment tables, plots, interpretations, summaries
for run_dir in sorted(out_root.glob("*/")):
    if "consolidated" in run_dir.name:
        continue
    cfg_path = run_dir / "config.json"
    label = run_dir.name
    if cfg_path.exists():
        label = _json.loads(cfg_path.read_text()).get("experiment_family", label)
    for sub in ["tables", "plots", "interpretations", "summaries"]:
        src = run_dir / sub
        if src.exists():
            shutil.copytree(str(src), str(pkg_dir / label / sub), dirs_exist_ok=True)

zip_path = shutil.make_archive(f"/content/{pkg_name}", "zip", root_dir=str(pkg_dir))
print(f"Package: {zip_path}")

drive_dir = Path("/content/drive/MyDrive/LinguoMT-AfricaS2T")
if drive_dir.exists():
    drive_dest = drive_dir / f"{pkg_name}.zip"
    shutil.copy2(zip_path, str(drive_dest))
    print(f"Drive backup: {drive_dest}")

if _in_colab:
    _colab_files.download(zip_path)
    print("Download triggered.")
else:
    print(f"Local package: {zip_path}")